# RQ2-v1 gradient-interference diagnostic

Read-only analysis of the four frozen Uniform/PureGeo seed 1/2 checkpoints. The notebook recalibrates BN locally, never updates weights, and uses the fixed 2,000-image validation subset.

## Secure repository checkout
Create a Kaggle secret named `github_token`. Enable Internet and select GPU T4 x2. `KAGGLE_API_TOKEN` is not needed because both prior outputs are mounted as notebook/dataset inputs.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Repository:', PROJECT_ROOT)

## Locate and validate the frozen RQ2-v1 inputs

In [ ]:
import importlib, json, torch
import rq2_anchor_placement
import rq2_gradient_interference
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_gradient_interference = importlib.reload(rq2_gradient_interference)

UNIFORM_INPUT = Path('/kaggle/input/datasets/dyhngg/100ep-extended')
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert UNIFORM_INPUT.exists(), f'Missing mounted Uniform-100 input: {UNIFORM_INPUT}'
assert RQ2_INPUT.exists(), f'Missing mounted RQ2-v1 input: {RQ2_INPUT}'
UNIFORM_ROOT = rq2_anchor_placement.find_uniform_100_root(
    UNIFORM_INPUT, '/kaggle/working/materialized-uniform-100-gradient'
)
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-v1-gradient'
)
CHECKPOINTS = rq2_gradient_interference.resolve_checkpoints(UNIFORM_ROOT, RQ2_ROOT)
print('Uniform root:', UNIFORM_ROOT)
print('RQ2 root:', RQ2_ROOT)
for key, path in CHECKPOINTS.items():
    print(key, path, f'{path.stat().st_size / 2**20:.1f} MiB')
print('CUDA devices:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Run the diagnostic
The two GPU workers process two checkpoints concurrently. Batch size 64 gives 32 minibatch-level conflict observations per checkpoint. If memory is tight, change it to 32.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-gradient-interference')
DATASET_ROOT = Path('/kaggle/working/data/cifar100')
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
started = time.perf_counter()
result = rq2_gradient_interference.run_gradient_interference(
    uniform_root=UNIFORM_ROOT,
    rq2_root=RQ2_ROOT,
    output_dir=OUTPUT_DIR,
    dataset_root=DATASET_ROOT,
    gpu_ids=GPU_IDS,
    batch_size=64,
)
print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')
print(json.dumps(result, indent=2))

## Inspect the decisive pairs and full summary

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
display(Markdown((OUTPUT_DIR / 'gradient_interference_summary.md').read_text()))
focus = pd.read_csv(OUTPUT_DIR / 'gradient_focus_pairs.csv')
display(focus.sort_values(['scope', 'seed', 'width_i', 'width_j']))
display(pd.read_csv(OUTPUT_DIR / 'gradient_norms_summary.csv'))
for filename in [
    'uniform_seed1_cosine.png', 'uniform_seed2_cosine.png',
    'puregeo_seed1_cosine.png', 'puregeo_seed2_cosine.png',
    'puregeo_minus_uniform_cosine.png',
]:
    display(Image(filename=str(OUTPUT_DIR / filename)))

## Validate and export

In [ ]:
REQUIRED = [
    'diagnostic_subset_ids.csv', 'resolved_diagnostic_config.yaml',
    'checkpoint_manifest.csv', 'gradient_pairwise_by_batch.csv',
    'gradient_pairwise_summary.csv', 'gradient_norms_by_batch.csv',
    'gradient_norms_summary.csv', 'gradient_puregeo_minus_uniform.csv',
    'gradient_focus_pairs.csv', 'gradient_interference_summary.md',
    'gradient_interference_complete.json', 'uniform_seed1_cosine.png',
    'uniform_seed2_cosine.png', 'puregeo_seed1_cosine.png',
    'puregeo_seed2_cosine.png', 'puregeo_minus_uniform_cosine.png',
]
missing = [name for name in REQUIRED if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Missing outputs: {missing}'
bundle_path = Path('/kaggle/working/rq2-gradient-interference.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download:', bundle_path, f'{bundle_path.stat().st_size / 2**20:.1f} MiB')
bundle_path